# Notebook 09: Deployment-Legal Shift Estimation

**Purpose**: Fit the shift context each v2 protocol is allowed to condition on, and
report the resulting shift taxonomy for the four real-arm datasets. This is the
input to Notebooks 10-14.

## Why this notebook exists

The v1 protocol suite (`src/selection/{random_select,similarity_select,label_diversity,feature_range,rule_diversity,counter_spurious}.py`)
is entirely a function of the *demonstration pool*. None of its members look at
the target domain, so none can adapt to a shift — which is a problem for a
project about OOD. `counter_spurious` is the only one that tries, and it needs a
`shift_col` naming the domain variable, which the extracted TableShift parquet
cache does not contain (TableShift consumes the domain splitter when it forms
train/test_id/test_ood). So on the real arm that protocol has no shift signal to
use as written.

`src/selection/shift_estimation.py` supplies that signal under a hard constraint:
**only what a deployed system can actually see** — labelled source data plus
*unlabelled* target inputs. No target label is ever read. That makes the
resulting protocols shift-type-agnostic: they are never told whether the shift is
covariate, prior, or mechanism, which is the setting the thesis cares about.

Three estimands, each mapping to a different failure mode:

| estimand | targets | trust condition |
|---|---|---|
| `s(x)`, `w(x)` — domain-discriminator score and density ratio | covariate shift, extrapolation | only when `DomainShift.auc` is meaningfully above 0.5 |
| `pi_T` — target label prior via BBSE (Lipton et al. 2018) | prior/label shift | only when the solve is interior (see the identifiability gate below) |
| per-feature drift (SMD / total variation) | diagnostic, and the replacement for `counter_spurious`'s missing `shift_col` | — |

**On the BBSE gate.** BBSE assumes `p(x|y)` is invariant and is badly biased when
that fails. An earlier version of this module shrank the estimate toward the
source prior, which taxed the three datasets where BBSE is *right* in order to
partially protect the one where it is wrong. `estimate_target_prior` instead
*gates* on two detectable failure signatures — a solution pinned to the simplex
boundary, or an ill-conditioned confusion matrix — and falls back to the source
prior when either fires. "The target prior is not identifiable from unlabelled
inputs here" is a reportable finding, not a failure.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Step 1: Fit the context

`scripts/prep_shift_context.py` does the fitting and caches one pickle per
dataset, so Notebooks 10-14 never refit it. Run it from the project root:

```bash
cd sata-project
PYTHONPATH=. python scripts/prep_shift_context.py --cache data/tableshift_raw_cache --out _screen_cache
```

It selects features by mutual information exactly as Notebook 01 does (top 12),
subsamples each split to 60k rows, then fits the discriminator, the gated BBSE
prior, and the drift table. `test_ood`'s label column is loaded only so later
notebooks can score predictions — nothing in the fitted context touches it.

The cell below loads the cache rather than refitting.

In [2]:
import pickle

import numpy as np
import pandas as pd

CACHE = PROJECT_ROOT / "_screen_cache"
DATASETS = ["brfss_diabetes", "acsincome", "acspubcov", "anes"]

ctx = {}
for ds in DATASETS:
    with open(CACHE / f"{ds}.pkl", "rb") as f:
        ctx[ds] = pickle.load(f)

pd.DataFrame([
    dict(dataset=ds,
         n_features=len(c["feats"]),
         n_train=len(c["train"]), n_test_id=len(c["test_id"]), n_test_ood=len(c["test_ood"]),
         prior_source=round(c["shift"].prior_source, 3),
         prior_target_true=round(c["true_prior_ood"], 3))
    for ds, c in ctx.items()
])

,dataset,n_features,n_train,n_test_id,n_test_ood,prior_source,prior_target_true
0,brfss_diabetes,12,60000,60000,60000,0.125,0.174
1,acsincome,12,60000,60000,60000,0.323,0.399
2,acspubcov,12,60000,60000,60000,0.224,0.637
3,anes,12,17112,2140,10719,0.696,0.617


## Step 2: The shift taxonomy

The two diagnostics dissociate, and that dissociation is the useful product: it
tells you *what kind* of shift each dataset actually poses, using only
information a deployed system would have.

- `domain_auc` — held-out ID-vs-target discriminator AUC. 0.5 means no detectable
  `p(x)` shift at all, in which case every `w(x)` is noise and any
  density-ratio protocol must degrade gracefully to its shift-agnostic twin
  (`DomainShift.detectable` is the flag, thresholded at 0.60).
- `prior_reliable` — whether the BBSE gate passed. Where it fails, the *raw*
  estimate is still reported so the failure is visible rather than hidden.

In [3]:
shift = pd.DataFrame([
    dict(dataset=ds,
         domain_auc=round(c["shift"].auc, 3),
         detectable=c["shift"].detectable,
         prior_source=round(c["shift"].prior_source, 3),
         prior_target_est=round(c["shift"].prior_target, 3),
         prior_target_bbse_raw=round(c["shift"].prior_target_raw, 3),
         prior_target_true=round(c["true_prior_ood"], 3),
         prior_reliable=c["shift"].prior_reliable,
         reason=c["shift"].prior_reason)
    for ds, c in ctx.items()
]).sort_values("domain_auc", ascending=False)

shift

,dataset,domain_auc,detectable,prior_source,prior_target_est,prior_target_bbse_raw,prior_target_true,prior_reliable,reason
1,acsincome,0.915,True,0.323,0.385,0.385,0.399,True,interior BBSE solution (cond=2.0)
2,acspubcov,0.808,True,0.224,0.224,1.000,0.637,False,BBSE solution on simplex boundary (raw=1.000); prior not identifiable
0,brfss_diabetes,0.668,True,0.125,0.208,0.208,0.174,True,interior BBSE solution (cond=19.8)
3,anes,0.557,False,0.696,0.663,0.663,0.617,True,interior BBSE solution (cond=2.8)


**Reading the table.** The four datasets land in four different regimes, which
is why protocol conclusions do not transfer between them:

- **`acsincome`** — high `domain_auc`, prior identifiable and close to truth:
  covariate-shift dominant.
- **`acspubcov`** — `domain_auc` high, but the prior is **not identifiable**: the
  BBSE solve pins to the simplex boundary, i.e. the observed target prediction
  distribution is not reachable by *any* reweighting of the source
  class-conditionals. That is precisely the `p(x|y)`-invariance violation the
  gate exists to catch, and the gate catches it.
- **`brfss_diabetes`** — moderate covariate shift, mild identifiable prior shift.
- **`anes`** — `domain_auc` near chance, so there is *no detectable `p(x)` shift*,
  yet the prior moves. Prior/concept shift with no covariate signal.

The practical consequence for Notebook 10: `importance_weighted` and
`shift_axis_coverage` have nothing to condition on for `anes`, and must degrade
to uniform rather than sample on noise.

In [4]:
# Supervised reference points, for calibrating expectations about what is
# achievable. A HistGradientBoosting model trained on the source split does
# degrade under these shifts -- which is the contrast the LLM results in
# Notebook 13 are measured against.
ref = pd.DataFrame([dict(dataset=ds, **{k: round(v, 3) for k, v in c["reference"].items()})
                    for ds, c in ctx.items()])
ref

,dataset,acc_id,acc_ood,auc_id,auc_ood,majority_id,majority_ood
0,brfss_diabetes,0.876,0.832,0.810,0.808,0.873,0.826
1,acsincome,0.815,0.803,0.884,0.893,0.679,0.601
2,acspubcov,0.797,0.608,0.768,0.796,0.776,0.637
3,anes,0.808,0.762,0.844,0.833,0.705,0.617


**This is the comparison that carries the RQ1 argument in Notebook 13.** The
supervised model is good in-domain and degrades out-of-domain — accuracy falls
where the label prior moves, while AUROC stays roughly flat. Hold onto the size
of that accuracy drop; the LLM does not reproduce it.

## Step 3: Per-feature drift and the shortcut candidates

`feature_drift` gives standardised mean difference for numeric features and total
variation distance for categoricals. `shift_proxy_features` then ranks by
`|corr(feature, label)| * drift(feature)` — a feature that both moves across
domains *and* predicts the label in-domain is exactly the shortcut a
demonstration set should be designed to break. This is the replacement for
`counter_spurious.find_spurious_proxy_features`'s unavailable `shift_col`.

In [5]:
for ds in DATASETS:
    c = ctx[ds]
    top = c["shift"].drift.head(4)[["feature", "drift", "statistic"]]
    print(f"{ds}  (proxy features: {', '.join(c['proxy_features'])})")
    print(top.to_string(index=False), end="\n\n")

brfss_diabetes  (proxy features: BMI5, HEALTH_COV, TOLDHI)
   feature    drift statistic
    INCOME 0.159733        tv
      BMI5 0.130913       smd
HEALTH_COV 0.128167        tv
  SMOKE100 0.068167        tv

acsincome  (proxy features: WKHP, POBP, AGEP)
feature    drift statistic
   POBP 0.652800        tv
   SCHL 0.086117        tv
   WKHP 0.069029       smd
   OCCP 0.068683        tv

acspubcov  (proxy features: ESR, PINCP, SCHL)
feature    drift statistic
   AGEP 0.653947       smd
    ESR 0.340933        tv
    FER 0.177300        tv
    MAR 0.140717        tv

anes  (proxy features: VCF0724, VCF0721, VCF0720)
feature    drift statistic
VCF0218 0.059591       smd
VCF9201 0.043479        tv
VCF9202 0.039754        tv
VCF0725 0.038829        tv



## Caveat carried forward

Shift estimates here use the **full** target input sample. A deployed system with
only a small unlabelled target batch would have noisier `w(x)` and `pi_T`; this
notebook does not quantify that sensitivity. Feature selection is the top 12 by
mutual information (Notebook 01's choice) — protocol rankings may shift with a
different feature budget.